### **Projeto Extensionista - Tópicos de Big Data em Python**

- **Faculdade Estácio de Sá RJ (UNESA)**
- **Aluno:** José Roberto Duarte Hegendorne  
- **Matrícula:** 202502214154  
- **Curso:** Presencial  
- **Professora:** Simone Gama  
- **Período:** 1º semestre 2026

**Tema:** Análise dos Dados do Sistema de Bilhetagem Eletrônica (RioCard)

**Fonte dos dados:** https://dadosabertos.rj.gov.br/dataset/setram_sbe

In [ ]:
import pandas as pd # Biblioteca para manipulação de dados
import numpy as np # Biblioteca para operações numéricas
import matplotlib.pyplot as plt # Biblioteca para visualização de dados
import seaborn as sns # Biblioteca para visualização de dados estatísticos
import re # Biblioteca para expressões regulares (limpeza de hash)
import os # Biblioteca para manipulação de arquivos
import zipfile # Biblioteca para manipulação de arquivos zip
from sklearn.linear_model import LinearRegression #
from sklearn.metrics import r2_score, mean_squared_error
import requests



In [ ]:
# Configurações de visualização
sns.set(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f6',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## **1. Carregamento dos Dados**

In [ ]:
# 0. Passo: Baixar os arquivos da pasta 'dados' do GitHub (raw URLs) para o Colab
Diretorio = os.path.join(".", "dados")
os.makedirs(Diretorio, exist_ok=True)

BASE_URL_RAW = "https://raw.githubusercontent.com/Heggendorn/facul/7d1dc57b88b1e3a444a84c7d7058252811e6fff8/dados/"


arquivos_zip = [
    "transacao_be_publico_2026_03_16_csv.zip",
    "transacao_be_publico_2026_03_17_csv.zip",
    "transacao_be_publico_2026_03_18_csv.zip",
]

for nome_arquivo in arquivos_zip:
    caminho_local = os.path.join(Diretorio, nome_arquivo)
    if not os.path.exists(caminho_local):
        url = f"{BASE_URL_RAW}/{nome_arquivo}"
        print(f"Baixando: {nome_arquivo}")
        resposta = requests.get(url)
        if resposta.status_code == 200:
            with open(caminho_local, 'wb') as f:
                f.write(resposta.content)
            print(f"  -> Salvo em {caminho_local} ({len(resposta.content)/1e6:.1f} MB)")
        else:
            print(f"  -> Erro ao baixar {nome_arquivo}: status {resposta.status_code}")
    else:
        print(f"Já existe localmente: {nome_arquivo}")


# 1. Define o caminho relativo para a pasta "dados"
# (já definido acima, mantido aqui por clareza)
# Diretorio = os.path.join(".", "dados")


def extrair_data_do_nome(nome_arquivo):
    """
    Extrai a data esperada a partir do nome do arquivo.
    Espera padrão: transacao_be_publico_AAAA_MM_DD_csv.zip
    Retorna um objeto date ou None se não encontrar.
    """
    match = re.search(r'(\d{4})_(\d{2})_(\d{2})', nome_arquivo)
    if match:
        ano, mes, dia = match.groups()
        try:
            return pd.Timestamp(int(ano), int(mes), int(dia)).date()
        except ValueError:
            return None
    return None


def Carregar_Dados(quantidade_linhas=None, arquivo_zip=r""):
    try:
        # Abre o arquivo ZIP em modo de leitura
        with zipfile.ZipFile(arquivo_zip, 'r') as z:
            # Lista os arquivos de dentro do ZIP e filtra para achar o arquivo CSV
            arquivos_internos = z.namelist()
            csv_encontrado = [f for f in arquivos_internos if f.endswith('.csv')]

            if not csv_encontrado:
                print(f"Nenhum CSV encontrado dentro do zip: {os.path.basename(arquivo_zip)}")
                return None

            # Abre o primeiro CSV encontrado dentro do ZIP sem precisar extrair para o disco
            with z.open(csv_encontrado[0]) as f:
                dados_df = pd.read_csv(f, encoding='utf-8-sig', sep=';', nrows=quantidade_linhas)
                return dados_df
    except Exception as e:
        print(f"Erro ao carregar os dados do arquivo {os.path.basename(arquivo_zip)}: {e}")
        return None


def Descontaminar_Datas(df, nome_arquivo):
    """
    Valida e limpa o dataframe de um arquivo individual.

    Fluxo:
    1. Extrai a data esperada do nome do arquivo (AAAA_MM_DD).
    2. Converte 'Data da Transação' (formato dd/mm/aaaa) para datetime.
    3. Calcula a data máxima presente no CSV.
    4. Se data_nome != data_maxima_csv -> descarta o arquivo inteiro (retorna None).
    5. Caso contrário, remove apenas as linhas cuja data não seja a data esperada.
    """
    if 'Data da Transação' not in df.columns:
        print(f"  -> Aviso: coluna 'Data da Transação' não encontrada em {nome_arquivo}. Arquivo descartado.")
        return None

    # 1. Passo: Extrai a data esperada do nome do arquivo
    data_nome = extrair_data_do_nome(nome_arquivo)
    if data_nome is None:
        print(f"  -> Aviso: não foi possível extrair data do nome '{nome_arquivo}'. Arquivo descartado.")
        return None

    # 2. Passo: Converte a coluna 'Data da Transação' (formato dd/mm/aaaa) para datetime
    datas_convertidas = pd.to_datetime(df['Data da Transação'], dayfirst=True, errors='coerce')

    if datas_convertidas.isna().all():
        print(f"  -> Aviso: não foi possível converter nenhuma data em {nome_arquivo}. Arquivo descartado.")
        return None

    # 3. Passo: Calcula a data máxima presente no CSV
    data_maxima_csv = datas_convertidas.max().date()

    # 4. Passo: Compara data do nome com a data máxima do CSV
    if data_nome != data_maxima_csv:
        print(f"  -> Data do nome ({data_nome}) != Data máxima do CSV ({data_maxima_csv}). "
              f"Arquivo '{nome_arquivo}' DESCARTADO por inconsistência.")
        return None

    # 5. Passo: Remove as linhas cuja data não seja a data esperada (data_nome)
    linhas_antes = len(df)
    mascara_validas = datas_convertidas.dt.date == data_nome
    df_filtrado = df[mascara_validas].copy()
    linhas_depois = len(df_filtrado)
    linhas_removidas = linhas_antes - linhas_depois
    pct_removido = (linhas_removidas / linhas_antes * 100) if linhas_antes else 0

    print(f"  -> Data validada: {data_nome} | "
          f"Removidas {linhas_removidas:,} de {linhas_antes:,} linhas ({pct_removido:.2f}%)")

    return df_filtrado


# Lista para armazenar o DataFrame de cada arquivo zipado
lista_dfs = []

# Verifica se a pasta "dados" existe no local do script
if os.path.exists(Diretorio):
    # Percorre todos os arquivos dentro do diretório "dados"
    for nome_arquivo in os.listdir(Diretorio):
        # Filtra para ler apenas arquivos que terminam com .zip
        if nome_arquivo.endswith('.zip'):
            caminho_completo = os.path.join(Diretorio, nome_arquivo)

            print(f"Processando: {nome_arquivo}")

            # Carrega os dados do arquivo zip atual
            df_individual = Carregar_Dados(quantidade_linhas=None, arquivo_zip=caminho_completo)

            # Se o arquivo foi lido com sucesso, valida e descontamina
            if df_individual is not None:
                df_individual = Descontaminar_Datas(df_individual, nome_arquivo)
                if df_individual is not None and len(df_individual) > 0:
                    lista_dfs.append(df_individual)

    # 2. Criar o DataFrame final (Bu_total) juntando todos os arquivos da lista
    if lista_dfs:
        Bu_total = pd.concat(lista_dfs, ignore_index=True)

        print(f"\nTotal de linhas carregadas de todos os ZIPs (após descontaminação): {len(Bu_total):,}")
        print(Bu_total.info())
    else:
        print("Nenhum arquivo ZIP com dados válidos foi encontrado na pasta 'dados'.")
else:
    print(f"A pasta 'dados' não foi encontrada. Certifique-se de criar a pasta chamada 'dados' ao lado do seu script Python.")

## **2. Seleção e Renomeação das Colunas**

In [ ]:
# Selecionando colunas relevantes
colunas = [
    "Data da Transação", "Nº Cartão", "Cartão Hash", "Descrição da Aplicação",
    "Sindicato", "Operadora", "Linha", "Sentido", "Nº Validador", 
    "Nº Carro", "Vl Linha", "Vl Trans", "Vl Subsídio"
]

Bu = Bu_total[colunas].copy()

# Renomeando colunas (padrão snake_case)
Bu.rename(columns={
    "Data da Transação": "Data_da_Transacao",
    "Nº Cartão": "N_Cartao",
    "Cartão Hash": "Cartao_Hash",
    "Descrição da Aplicação": "Descricao_da_Aplicacao",
    "Nº Validador": "N_Validador",
    "Nº Carro": "N_Carro",
    "Vl Linha": "Vl_Linha",
    "Vl Trans": "Vl_Trans",
    "Vl Subsídio": "Vl_Subsidio"
}, inplace=True)

## **3. Tratamento dos Dados**

In [ ]:
# Tratamento do campo Sentido
Bu["Sentido"] = Bu["Sentido"].fillna(0)
Bu["Sentido"] = Bu["Sentido"].round().astype(int)
Bu["Sentido"] = Bu["Sentido"].map({0: "Desconhecido", 1: "Ida", 2: "Volta"})

## **Tratamento de Valores Monetários**

In [ ]:
# Vl_Subsidio
Bu["Vl_Subsidio"] = Bu["Vl_Subsidio"].fillna(0)
Bu['Vl_Subsidio'] = (
    Bu['Vl_Subsidio']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .astype(float)
)

# Vl_Linha e Vl_Trans
Bu["Vl_Linha"] = Bu["Vl_Linha"].str.replace(",", ".").astype(float)
Bu["Vl_Trans"] = Bu["Vl_Trans"].str.replace(",", ".").astype(float)

## **Tratamento de Data e Hora**

In [ ]:
# Conversão de data
Bu["Data_da_Transacao"] = pd.to_datetime(Bu["Data_da_Transacao"],dayfirst=True,errors='coerce') 
Bu['hora'] = Bu['Data_da_Transacao'].dt.hour
Bu['data'] = Bu['Data_da_Transacao'].dt.date

## **Limpeza de Hash e Outros**

In [ ]:
def limpar_hash(texto):
    """Substitui caracteres especiais por underscore (_), mantendo letras e números"""
    if pd.isna(texto) or texto is None:   # Correção aqui!
        return ""
    
    # Converte para string
    texto = str(texto)
    
    # Substitui tudo que não for letra ou número por "_"
    texto_limpo = re.sub(r'[^a-zA-Z0-9]',lambda m: str(ord(m.group())),texto)
    
    # Remove underscores múltiplos (___ vira _)
    texto_limpo = re.sub(r'_+', '_', texto_limpo)
    
    # Remove _ do início e fim
    texto_limpo = texto_limpo.strip('_')
    
    return texto_limpo


Bu['Cartao_Hash'] = Bu['Cartao_Hash'].apply(limpar_hash)
Bu['eh_qrcode'] = Bu['N_Cartao'].astype(str) == 'xxxxxx'

## **Removendo registros inválidos**

In [ ]:
print(f"Antes da limpeza: {len(Bu):,} registros")

# Remoções
Bu = Bu[Bu["Vl_Linha"] > 0].copy()                    # Remove Vl_Linha = 0
Bu = Bu[Bu['Descricao_da_Aplicacao'] != '820-BARCAS MORADOR'].copy()
Bu = Bu.dropna(subset=["Linha", "Operadora", "Vl_Linha"]).copy()

print(f"Depois da limpeza: {len(Bu):,} registros")

## **Remoção de duplicatas**

In [ ]:
# Remoção de duplicatas
print(f"Duplicatas encontradas: {Bu.duplicated().sum()}")
Bu = Bu.drop_duplicates().reset_index(drop=True)
print(f"Após remoção de duplicatas: {len(Bu):,} registros")

## **Classificação de Perfil (Melhorada)**

In [ ]:
# Define as condições lógicas
condicoes = [
    Bu["Vl_Trans"] == 0,
    (Bu["Vl_Subsidio"] > 0)
    & (abs(Bu["Vl_Trans"] + Bu["Vl_Subsidio"] - Bu["Vl_Linha"]) < 0.1),
    (Bu["Vl_Subsidio"] > 0)
    & (abs(Bu["Vl_Trans"] + Bu["Vl_Subsidio"] - Bu["Vl_Linha"]) >= 0.1),
]

# Define os resultados para cada condição
resultados = ["Gratuidade", "Subsidiado", "Integracao"]

# O padrão (else) será 'Pagamento_Integral'
Bu["Perfil"] = np.select(condicoes, resultados, default="Pagamento_Integral")
Bu['Diferenca'] = Bu['Vl_Linha'] - Bu['Vl_Trans']

## **4. Verificação Final da Base Tratada**

In [ ]:
print("=== INFORMAÇÕES FINAIS DA BASE ===")
print(Bu.info())
print("\n=== VALORES NULOS ===")
print(Bu.isnull().sum())
print("\n=== DUPLICATAS ===")
print(Bu.duplicated().sum())

Bu.head()

---



## **5. ANÁLISE DESCRITIVA**

  ### 5.1  Estatísticas descritivas gerais e totais

In [ ]:
# 5.1 - Estatísticas descritivas gerais e totais

# VARIAVEIS QUANTITATIVAS
print("=== ESTATÍSTICAS DESCRITIVAS GERAIS ===")
print(Bu[['Vl_Linha', 'Vl_Trans', 'Vl_Subsidio']].describe().round(2))
print("\n=== TOTAL GERAL ===")
print(f"Total de viagens: {len(Bu):,}")
print(f"Valor total pago pelos usuários: R$ {Bu['Vl_Trans'].sum():,.2f}")
print(f"Valor total subsidiado pelo governo: R$ {Bu['Vl_Subsidio'].sum():,.2f}")
print(f"Valor total das tarifas cheias: R$ {Bu['Vl_Linha'].sum():,.2f}")

# === NOVO: Evolução diária (base agora cobre múltiplos dias) ===
resumo_diario = Bu.groupby('data').agg(
    Total_Viagens=('Data_da_Transacao', 'count'),
    Total_Pago=('Vl_Trans', 'sum'),
    Total_Subsidio=('Vl_Subsidio', 'sum'),
    Total_Tarifa_Cheia=('Vl_Linha', 'sum')
).reset_index()

print("\n=== RESUMO POR DIA ===")
print(resumo_diario.round(2))

dia_maior_volume = resumo_diario.loc[resumo_diario['Total_Viagens'].idxmax()]
dia_menor_volume = resumo_diario.loc[resumo_diario['Total_Viagens'].idxmin()]
print(f"\nDia com maior volume de viagens: {dia_maior_volume['data']} ({dia_maior_volume['Total_Viagens']:,.0f} viagens)")
print(f"Dia com menor volume de viagens: {dia_menor_volume['data']} ({dia_menor_volume['Total_Viagens']:,.0f} viagens)")

### 5.2 - DISTRIBUIÇÃO POR PERFIL

In [ ]:
#   VARIAVEIS CATEGÓRICAS 
perfil = Bu['Perfil'].value_counts() # Quantidade de cada perfil
perfil_pct = Bu['Perfil'].value_counts(normalize=True) * 100 # Porcentagem de cada perfil em relação ao total
print("=== DISTRIBUIÇÃO POR PERFIL ===")
print(pd.DataFrame({
    'Quantidade': perfil,
    'Porcentagem (%)': perfil_pct.round(2)
}))

# === NOVO: Distribuição por perfil, dia a dia ===
perfil_por_dia = Bu.groupby(['data', 'Perfil']).size().unstack(fill_value=0)
perfil_por_dia_pct = perfil_por_dia.div(perfil_por_dia.sum(axis=1), axis=0) * 100

print("\n=== DISTRIBUIÇÃO POR PERFIL E DIA (Quantidade) ===")
print(perfil_por_dia)
print("\n=== DISTRIBUIÇÃO POR PERFIL E DIA (% dentro de cada dia) ===")
print(perfil_por_dia_pct.round(2))

## 5.2 - análise descritiva


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Análise Descritiva - Bilhetagem Eletrônica RJ', fontsize=16, fontweight='bold')

# 1. Volume por hora — agora como média por dia, com uma linha por dia (em vez de soma agregada)
por_hora_dia = Bu.groupby(['data', 'hora']).size().unstack(level=0, fill_value=0)
for dia in por_hora_dia.columns:
    axes[0,0].plot(por_hora_dia.index, por_hora_dia[dia], marker='o', alpha=0.7, label=str(dia))
axes[0,0].set_title('Volume de Viagens por Hora do Dia (por data)')
axes[0,0].set_xlabel('Hora')
axes[0,0].set_ylabel('Quantidade de Viagens')
axes[0,0].legend(title='Data', fontsize=8)

# 2. Distribuição Vl_Trans (mantido agregado - faz sentido olhar a distribuição geral)
sns.histplot(Bu['Vl_Trans'], bins=30, kde=True, ax=axes[0,1], color='#ff7f0e')
axes[0,1].set_title('Distribuição do Valor Pago pelo Usuário (Vl_Trans)')
axes[0,1].set_xlabel('Valor (R$)')

# 3. Proporção por Perfil (mantido agregado)
axes[1,0].pie(perfil.values, labels=perfil.index, autopct='%1.1f%%', startangle=90)
axes[1,0].set_title('Composição por Perfil de Passageiro')

# 4. Boxplot por Perfil (mantido agregado)
sns.boxplot(data=Bu, x='Perfil', y='Vl_Trans', ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Valor Pago por Perfil')
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('imgs/analise_descritiva4graficos.png', bbox_inches='tight', dpi=150)
plt.show()

## **5.3  função para categorizar as horas em momentos do dia**
- Distribuição de Frequência: Volume de Viagens por Turno Operacional

In [ ]:
# 1. Passo:  função para categorizar as horas em momentos do dia
def mapear_turno(hora_bruta):
    if 6 <= hora_bruta <= 8:
        return '1. Pico Manhã (06h-08h)'
    elif 16 <= hora_bruta <= 19:
        return '3. Pico Tarde (16h-19h)'
    elif 9 <= hora_bruta <= 15:
        return '2. Entre-Pico Diurno (9h-15h)'
    else:
        return '4. Noite/Madrugada'

# 2. Passo: Aplicar a função no DataFrame criando a nova coluna 'Turno'
Bu['Turno'] = Bu['hora'].apply(mapear_turno)

# 3. Passo: Calcular o volume de viagens e o subsídio por turno (agregado - mantido para visão geral)
resumo_turnos = Bu.groupby('Turno').agg(
    Total_Viagens=('Data_da_Transacao', 'count'),
    Subsidio_Total=('Vl_Subsidio', 'sum')
).reset_index()

# 4. Passo: Plotar gráfico de linha mostrando a progressão por turno
plt.figure(figsize=(10, 5))
plt.plot(resumo_turnos['Turno'], resumo_turnos['Total_Viagens'],
         marker='o', linewidth=2, color='#1f77b4')

# Destaca os pontos de pico com cor diferente e anota o valor de cada turno
for i, row in resumo_turnos.iterrows():
    cor_ponto = '#d95f02' if 'Pico' in row['Turno'] else '#7570b3'
    plt.scatter(row['Turno'], row['Total_Viagens'], color=cor_ponto, s=80, zorder=3)
    plt.annotate(f"{row['Total_Viagens']:,.0f}",
                  (row['Turno'], row['Total_Viagens']),
                  textcoords="offset points", xytext=(0, 10),
                  ha='center', fontsize=9, fontweight='bold')

plt.title('Distribuição de Frequência: Volume de Viagens por Turno Operacional (Geral)', fontsize=14, fontweight='bold')
plt.xlabel('Turnos Operacionais do Transporte Público')
plt.ylabel('Quantidade Absoluta de Viagens')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('imgs/frequencia_sindicato_linha.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

# === Estatísticas descritivas do gráfico geral por turno ===
total_viagens_geral = resumo_turnos['Total_Viagens'].sum()
resumo_turnos['Pct_do_Total'] = (resumo_turnos['Total_Viagens'] / total_viagens_geral * 100).round(2)
turno_pico = resumo_turnos.loc[resumo_turnos['Total_Viagens'].idxmax()]
turno_minimo = resumo_turnos.loc[resumo_turnos['Total_Viagens'].idxmin()]

print("=== ESTATÍSTICAS DESCRITIVAS - VOLUME POR TURNO (GERAL) ===")
print(resumo_turnos[['Turno', 'Total_Viagens', 'Pct_do_Total', 'Subsidio_Total']])
print(f"\nMédia de viagens por turno: {resumo_turnos['Total_Viagens'].mean():,.0f}")
print(f"Desvio padrão entre turnos: {resumo_turnos['Total_Viagens'].std():,.0f}")
print(f"Turno de maior movimento: {turno_pico['Turno']} ({turno_pico['Total_Viagens']:,.0f} viagens | {turno_pico['Pct_do_Total']:.2f}% do total)")
print(f"Turno de menor movimento: {turno_minimo['Turno']} ({turno_minimo['Total_Viagens']:,.0f} viagens | {turno_minimo['Pct_do_Total']:.2f}% do total)")


# === NOVO: Distribuição de frequência por turno, comparando os dias ===
resumo_turnos_dia = Bu.groupby(['data', 'Turno']).size().unstack(fill_value=0)

ax = resumo_turnos_dia.plot(
    kind='bar',
    figsize=(12, 6),
    color=['#1b9e77', '#d95f02', '#7570b3', '#e7298a'],
    alpha=0.85
)

# Anota o valor de cada barra
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', fontsize=7, padding=2)

plt.title('Distribuição de Frequência por Turno Operacional - Comparação entre Dias', fontsize=14, fontweight='bold')
plt.xlabel('Data')
plt.ylabel('Quantidade de Viagens')
plt.xticks(rotation=0)
plt.legend(title='Turno', fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('imgs/frequencia_turnos_por_dia.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

# === Estatísticas descritivas do gráfico por dia e turno ===
print("\n=== ESTATÍSTICAS DESCRITIVAS - VOLUME POR TURNO E DIA ===")
print(resumo_turnos_dia)

print("\n=== PERCENTUAL DE CADA TURNO DENTRO DE CADA DIA ===")
print(resumo_turnos_dia.div(resumo_turnos_dia.sum(axis=1), axis=0).mul(100).round(2))

print("\n=== MÉDIA E DESVIO POR TURNO (ENTRE OS DIAS) ===")
estatisticas_turno = resumo_turnos_dia.agg(['mean', 'std']).round(1)
print(estatisticas_turno)

# Turno mais consistente (menor variação relativa) vs mais variável
cv_turno = (estatisticas_turno.loc['std'] / estatisticas_turno.loc['mean'] * 100).round(2)
print("\n=== COEFICIENTE DE VARIAÇÃO (%) POR TURNO ===")
print(cv_turno)
print(f"\nTurno mais consistente entre os dias: {cv_turno.idxmin()} (CV = {cv_turno.min():.2f}%)")
print(f"Turno mais variável entre os dias: {cv_turno.idxmax()} (CV = {cv_turno.max():.2f}%)")

### **Distribuição de Frequência por Turnos Operacionais**

**O que este código faz?**
Ele pega o horário exato de cada um dos milhares de registros de passagens e os joga dentro de "baldes" de tempo: o horário em que as pessoas vão para o trabalho (Pico da Manhã), o horário de retorno (Pico da Tarde), o horário intermediário (Entre-Pico) e o turno da noite. Depois, conta quantas viagens aconteceram em cada um desses blocos.

**Por que isso importa para o projeto?**
Em sistemas de transporte, analisar horas isoladas gera ruído visual. Agrupar em turnos operacionais transforma dados brutos de Big Data em planejamento urbano. O gráfico destaca visualmente (utilizando cores diferenciadas) a concentração massiva da população nos horários de pico, que é exatamente onde o sistema de bilhetagem sofre a maior pressão de processamento de dados e onde a frota precisa de reforço.

### 5.4 correlações observadas nas variaveis quantitativas

In [ ]:
print("\n=== PRINCIPAIS CORRELAÇÕES OBSERVADAS ===")

# 1. Passo: Selecionar apenas as colunas com números para calcular a correlação de forma segura
colunas_numericas = Bu[['Vl_Linha', 'Vl_Trans', 'Vl_Subsidio', 'hora']]

# 2. Passo: Criar a matriz de correlação
correlacao = colunas_numericas.corr()

# 3. Passo: Exibir as correlações em relação ao valor pago pelo usuário (Vl_Trans)
print("Correlação com Vl_Trans (Valor Pago pelo Usuário):")
print(correlacao['Vl_Trans'].sort_values(ascending=False).round(3))

In [ ]:
# 1. Passo: Filtrar apenas passageiros do perfil 'Subsidiado' que geram custo ao governo
dados_subsidio = Bu[(Bu['Perfil'] == 'Subsidiado') & (Bu['Vl_Subsidio'] > 0)].copy()

# 2. Passo: Definir a variável explicativa (X = Tarifa Cheia) e a dependente (y = Subsídio pago)
X_reg = dados_subsidio[['Vl_Linha']]
y_reg = dados_subsidio['Vl_Subsidio']

# 3. Passo: Treinar o modelo de Regressão Linear
modelo_financeiro = LinearRegression()
modelo_financeiro.fit(X_reg, y_reg)

# 4. Passo: Calcular as previsões e métricas de erro
y_pred_reg = modelo_financeiro.predict(X_reg)
r2_fin = r2_score(y_reg, y_pred_reg)
rmse_fin = np.sqrt(mean_squared_error(y_reg, y_pred_reg))

# 5. Passo: Exibir os resultados técnicos
print("=== REGRESSÃO: VALOR DA LINHA × SUBSÍDIO (PERFIL SUBSIDIADO) ===")
print(f"Número de dias na base: {Bu['data'].nunique()}")
print(f"Amostra utilizada: {len(dados_subsidio):,} registros")
print(f"Coeficiente angular (Inclinação): {modelo_financeiro.coef_[0]:.4f}")
print(f"Intercepto (Constante): {modelo_financeiro.intercept_:.4f}")
print(f"R² (Poder de Explicação): {r2_fin*100:.1f}%")
print(f"Erro Médio (RMSE): R$ {rmse_fin:.2f}")

# 6. Passo: Gerar o gráfico para a apresentação
plt.figure(figsize=(10, 5))
plt.scatter(X_reg.sample(min(2000, len(X_reg)), random_state=42), 
            y_reg.sample(min(2000, len(y_reg)), random_state=42), 
            color='teal', alpha=0.5, label='Amostra de Transações Reais')
plt.plot(X_reg, y_pred_reg, color='red', linewidth=3, label='Linha de Tendência (Regressão)')
plt.title('Regressão Linear: Como a Tarifa da Linha Determina o Gasto com Subsídio', fontsize=14, fontweight='bold')
plt.xlabel('Valor Total da Linha (Tarifa Cheia em R$)')
plt.ylabel('Valor do Subsídio Pago pelo Governo (R$)')
plt.legend()
plt.grid(True, linestyle='--')
plt.tight_layout()
plt.savefig('imgs/regressao_subsidio.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# === NOVO: Identificação de Outliers - Comparação entre Perfis (Integral x Subsidiado) por Dia ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Identificação de Outliers - Comparação entre Perfis de Pagamento por Dia', fontsize=15, fontweight='bold')

# Filtra apenas os perfis de interesse para a comparação
perfis_interesse = ['Pagamento_Integral', 'Subsidiado']
Bu_comparacao = Bu[Bu['Perfil'].isin(perfis_interesse)].copy()

# 1. Painel - Vl_Trans (Valor Pago pelo Usuário)
sns.boxplot(
    data=Bu_comparacao,
    x='data',
    y='Vl_Trans',
    hue='Perfil',
    palette='Set2',
    fliersize=4,
    flierprops={"markerfacecolor": "red", "markeredgecolor": "red", "alpha": 0.5},
    ax=axes[0]
)
axes[0].set_title('Valor Pago pelo Usuário (Vl_Trans)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Data')
axes[0].set_ylabel('Vl_Trans (R$)')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
axes[0].legend(title='Perfil')

# 2. Painel - Vl_Linha (Tarifa Cheia)
sns.boxplot(
    data=Bu_comparacao,
    x='data',
    y='Vl_Linha',
    hue='Perfil',
    palette='Set2',
    fliersize=4,
    flierprops={"markerfacecolor": "red", "markeredgecolor": "red", "alpha": 0.5},
    ax=axes[1]
)
axes[1].set_title('Tarifa Cheia da Linha (Vl_Linha)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Data')
axes[1].set_ylabel('Vl_Linha (R$)')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
axes[1].legend(title='Perfil')

plt.tight_layout()
plt.savefig('imgs/outliers_perfil_dia.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

---
##  Análise separando os dois grupos, os que tem gratuidade e os que não tem



In [ ]:
# === Criação do agrupamento binário: Pagamento Integral x Com Subsídio ===
Bu['Grupo_Pagamento'] = Bu['Perfil'].apply(
    lambda x: 'Pagamento_Integral' if x == 'Pagamento_Integral' else 'Com_Subsidio'
)# Cria uma nova coluna 'Grupo_Pagamento' onde 'Pagamento_Integral' permanece igual e os outros perfis são agrupados como 'Com_Subsidio'

print("=== DISTRIBUIÇÃO DO AGRUPAMENTO BINÁRIO ===")
print(Bu['Grupo_Pagamento'].value_counts())
print(Bu['Grupo_Pagamento'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# === ANÁLISE DESCRITIVA: Pagamento Integral x Com Subsídio ===
print("=== ESTATÍSTICAS DESCRITIVAS POR GRUPO DE PAGAMENTO ===")
print(Bu.groupby('Grupo_Pagamento')[['Vl_Linha', 'Vl_Trans', 'Vl_Subsidio']].describe().round(2))

print("\n=== TOTAIS POR GRUPO ===")
resumo_grupo = Bu.groupby('Grupo_Pagamento').agg(
    Total_Viagens=('Data_da_Transacao', 'count'),
    Total_Pago=('Vl_Trans', 'sum'),
    Total_Subsidio=('Vl_Subsidio', 'sum'),
    Total_Tarifa_Cheia=('Vl_Linha', 'sum'),
    Media_Vl_Trans=('Vl_Trans', 'mean'),
    Media_Vl_Linha=('Vl_Linha', 'mean')
).round(2)
print(resumo_grupo)

# Gráfico de barras comparando médias
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Análise Descritiva - Pagamento Integral x Com Subsídio', fontsize=15, fontweight='bold')

resumo_grupo['Total_Viagens'].plot(kind='bar', ax=axes[0], color=['#1f77b4', '#ff7f0e'])
axes[0].set_title('Quantidade de Viagens por Grupo')
axes[0].set_ylabel('Quantidade de Viagens')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

resumo_grupo[['Media_Vl_Trans', 'Media_Vl_Linha']].plot(kind='bar', ax=axes[1], color=['#2ca02c', '#9467bd'])
axes[1].set_title('Valor Médio Pago vs Tarifa Cheia')
axes[1].set_ylabel('Valor (R$)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('imgs/descritiva_grupo_pagamento.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

In [ ]:
# === REGRESSÃO LINEAR: Vl_Linha x Vl_Trans, separada por Grupo de Pagamento ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Regressão Linear: Tarifa Cheia x Valor Pago - por Grupo', fontsize=15, fontweight='bold')

resultados_regressao = {}

for i, grupo in enumerate(['Pagamento_Integral', 'Com_Subsidio']):
    dados_grupo = Bu[Bu['Grupo_Pagamento'] == grupo].copy()

    X = dados_grupo[['Vl_Linha']]
    y = dados_grupo['Vl_Trans']

    modelo = LinearRegression()
    modelo.fit(X, y)
    y_pred = modelo.predict(X)

    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))

    resultados_regressao[grupo] = {
        'coeficiente': modelo.coef_[0],
        'intercepto': modelo.intercept_,
        'r2': r2,
        'rmse': rmse,
        'n': len(dados_grupo)
    }

    cor = '#1f77b4' if grupo == 'Pagamento_Integral' else '#d62728'

    amostra_x = X.sample(min(2000, len(X)), random_state=42)
    amostra_y = y.sample(min(2000, len(y)), random_state=42)

    axes[i].scatter(amostra_x, amostra_y, alpha=0.3, s=8, color=cor, label='Transações')
    axes[i].plot(X, y_pred, color='black', linewidth=2.5, label='Linha de Tendência')
    axes[i].set_title(f'{grupo}\nR² = {r2*100:.1f}% | RMSE = R$ {rmse:.2f}')
    axes[i].set_xlabel('Vl_Linha (Tarifa Cheia)')
    axes[i].set_ylabel('Vl_Trans (Pago)')
    axes[i].legend()
    axes[i].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('imgs/regressao_grupo_pagamento.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

print("=== RESULTADOS DA REGRESSÃO POR GRUPO ===")
for grupo, res in resultados_regressao.items():
    print(f"\n{grupo} (n={res['n']:,}):")
    print(f"  Coeficiente angular: {res['coeficiente']:.4f}")
    print(f"  Intercepto: {res['intercepto']:.4f}")
    print(f"  R²: {res['r2']*100:.1f}%")
    print(f"  RMSE: R$ {res['rmse']:.2f}")

In [ ]:
# === DIVISÃO DE FREQUÊNCIA: Volume por Turno, separado por Grupo de Pagamento ===
resumo_turno_grupo = Bu.groupby(['Turno', 'Grupo_Pagamento']).size().unstack(fill_value=0)

plt.figure(figsize=(11, 5))
resumo_turno_grupo.plot(kind='bar', ax=plt.gca(), color=['#1f77b4', '#d62728'], alpha=0.85)

plt.title('Distribuição de Frequência por Turno - Pagamento Integral x Com Subsídio', fontsize=14, fontweight='bold')
plt.xlabel('Turnos Operacionais do Transporte Público')
plt.ylabel('Quantidade de Viagens')
plt.xticks(rotation=15)
plt.legend(title='Grupo')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('imgs/frequencia_grupo_pagamento.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

print("=== TABELA DE FREQUÊNCIA POR TURNO E GRUPO ===")
print(resumo_turno_grupo)
print("\n=== PERCENTUAL DENTRO DE CADA TURNO ===")
print(resumo_turno_grupo.div(resumo_turno_grupo.sum(axis=1), axis=0).mul(100).round(2))

In [ ]:
# === DIVISÃO DE FREQUÊNCIA: Faixas de 2 em 2 horas, comparando os dias (20/21/22) ===

# 1. Passo: Criar a coluna de faixa horária (intervalos de 2 em 2 horas)
def mapear_faixa_2h(hora_bruta):
    inicio = (hora_bruta // 2) * 2
    fim = inicio + 2
    return f"{inicio:02d}:00 - {fim:02d}:00"

Bu['Faixa_2h'] = Bu['hora'].apply(mapear_faixa_2h)

# 2. Passo: Garantir a ordem correta das faixas no eixo X
ordem_faixas = [f"{h:02d}:00 - {h+2:02d}:00" for h in range(0, 24, 2)]
Bu['Faixa_2h'] = pd.Categorical(Bu['Faixa_2h'], categories=ordem_faixas, ordered=True)

# 3. Passo: Calcular a contagem de viagens por faixa horária e por dia
resumo_faixas_dia = Bu.groupby(['Faixa_2h', 'data']).size().unstack(fill_value=0)

# 4. Passo: Plotar gráfico de barras agrupadas (uma cor por dia, lado a lado)
ax = resumo_faixas_dia.plot(
    kind='bar',
    figsize=(14, 6),
    colormap='tab10',
    alpha=0.85
)

plt.title('Distribuição de Frequência: Volume de Viagens por Faixa de 2 Horas - Comparação entre Dias', 
          fontsize=14, fontweight='bold')
plt.xlabel('Faixa Horária')
plt.ylabel('Quantidade de Viagens')
plt.xticks(rotation=45)
plt.legend(title='Data')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('imgs/frequencia_faixa_2h_por_dia.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

# 5. Passo: Exibir a tabela com os valores
print("=== VOLUME DE VIAGENS POR FAIXA DE 2H E POR DIA ===")
print(resumo_faixas_dia)

In [ ]:
# === DIVISÃO DE FREQUÊNCIA: Faixas de 2 em 2 horas, comparando os dias (20/21/22) + linha de média ===

# 1. Passo: Criar a coluna de faixa horária (intervalos de 2 em 2 horas)
def mapear_faixa_2h(hora_bruta):
    inicio = (hora_bruta // 2) * 2
    fim = inicio + 2
    return f"{inicio:02d}:00 - {fim:02d}:00"

Bu['Faixa_2h'] = Bu['hora'].apply(mapear_faixa_2h)

# 2. Passo: Garantir a ordem correta das faixas no eixo X
ordem_faixas = [f"{h:02d}:00 - {h+2:02d}:00" for h in range(0, 24, 2)]
Bu['Faixa_2h'] = pd.Categorical(Bu['Faixa_2h'], categories=ordem_faixas, ordered=True)

# 3. Passo: Calcular a contagem de viagens por faixa horária e por dia
resumo_faixas_dia = Bu.groupby(['Faixa_2h', 'data']).size().unstack(fill_value=0)

# 4. Passo: Calcular a média entre os dias, para cada faixa horária
media_por_faixa = resumo_faixas_dia.mean(axis=1)

# 5. Passo: Plotar gráfico de barras agrupadas (uma cor por dia, lado a lado)
ax = resumo_faixas_dia.plot(
    kind='bar',
    figsize=(14, 6),
    colormap='tab10',
    alpha=0.85
)

# 6. Passo: Adicionar a linha de média (entre os dias) de cada faixa horária
plt.plot(range(len(media_por_faixa)), media_por_faixa.values,
         color='black', linewidth=2.5, linestyle='--', marker='o',
         label='Média entre os dias (por faixa)')

plt.title('Distribuição de Frequência: Volume de Viagens por Faixa de 2 Horas - Comparação entre Dias', 
          fontsize=14, fontweight='bold')
plt.xlabel('Faixa Horária')
plt.ylabel('Quantidade de Viagens')
plt.xticks(rotation=45)
plt.legend(title='Data / Referência')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('imgs/frequencia_faixa_2h_por_dia.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

# 7. Passo: Exibir a tabela com os valores
print("=== VOLUME DE VIAGENS POR FAIXA DE 2H E POR DIA ===")
print(resumo_faixas_dia)
print("\n=== MÉDIA ENTRE OS DIAS, POR FAIXA HORÁRIA ===")
print(media_por_faixa.round(1))

# === ANÁLISE DESCRITIVA: Média diária de passageiros (viagens) ===
viagens_por_dia = Bu.groupby('data').size()
media_diaria_viagens = viagens_por_dia.mean()

print("\n=== TOTAL DE VIAGENS POR DIA ===")
print(viagens_por_dia)

print(f"\n=== MÉDIA DIÁRIA DE PASSAGEIROS (VIAGENS) ===")
print(f"Média diária: {media_diaria_viagens:,.0f} viagens/dia")
print(f"Desvio-padrão entre os dias: {viagens_por_dia.std():,.0f}")
print(f"Dia com maior volume: {viagens_por_dia.idxmax()} ({viagens_por_dia.max():,} viagens)")
print(f"Dia com menor volume: {viagens_por_dia.idxmin()} ({viagens_por_dia.min():,} viagens)")

In [ ]:
# === ANÁLISE DESCRITIVA: Média diária de passageiros (viagens) ===

# 1. Passo: Total de viagens por dia
viagens_por_dia = Bu.groupby('data').size()

# 2. Passo: Calcular a média diária
media_diaria_viagens = viagens_por_dia.mean()

print("=== TOTAL DE VIAGENS POR DIA ===")
print(viagens_por_dia)

print(f"\n=== MÉDIA DIÁRIA DE PASSAGEIROS (VIAGENS) ===")
print(f"Média diária: {media_diaria_viagens:,.0f} viagens/dia")
print(f"Desvio-padrão entre os dias: {viagens_por_dia.std():,.0f}")
print(f"Dia com maior volume: {viagens_por_dia.idxmax()} ({viagens_por_dia.max():,} viagens)")
print(f"Dia com menor volume: {viagens_por_dia.idxmin()} ({viagens_por_dia.min():,} viagens)")

In [ ]:
# === ANÁLISE DESCRITIVA: Média diária de passageiros + Totais financeiros ===

# 1. Passo: Total de viagens por dia
viagens_por_dia = Bu.groupby('data').size()

# 2. Passo: Calcular a média diária
media_diaria_viagens = viagens_por_dia.mean()

print("=== TOTAL DE VIAGENS POR DIA ===")
print(viagens_por_dia)

print(f"\n=== MÉDIA DIÁRIA DE PASSAGEIROS (VIAGENS) ===")
print(f"Média diária: {media_diaria_viagens:,.0f} viagens/dia")
print(f"Desvio-padrão entre os dias: {viagens_por_dia.std():,.0f}")
print(f"Dia com maior volume: {viagens_por_dia.idxmax()} ({viagens_por_dia.max():,} viagens)")
print(f"Dia com menor volume: {viagens_por_dia.idxmin()} ({viagens_por_dia.min():,} viagens)")

# 3. Passo: Valor total arrecadado (Vl_Trans = o que efetivamente entrou via passageiro)
valor_total_arrecadado = Bu['Vl_Trans'].sum()

# 4. Passo: Valor total pago integralmente (apenas Grupo_Pagamento == 'Pagamento_Integral')
valor_pago_integral = Bu.loc[Bu['Grupo_Pagamento'] == 'Pagamento_Integral', 'Vl_Trans'].sum()

# 5. Passo: Valor total subsidiado pelo governo
valor_total_subsidiado = Bu['Vl_Subsidio'].sum()

# 6. Passo: Valor pago pelos passageiros do grupo com subsídio (Vl_Trans deles)
valor_pago_com_subsidio = Bu.loc[Bu['Grupo_Pagamento'] == 'Com_Subsidio', 'Vl_Trans'].sum()

# 7. Passo: Valor total das tarifas cheias (referência - quanto custaria sem nenhum subsídio)
valor_total_tarifa_cheia = Bu['Vl_Linha'].sum()

print("\n=== RESUMO FINANCEIRO GERAL ===")
print(f"Valor total arrecadado (Vl_Trans): R$ {valor_total_arrecadado:,.2f}")
print(f"  - Pago integralmente (sem subsídio): R$ {valor_pago_integral:,.2f}")
print(f"  - Pago pelo grupo com subsídio: R$ {valor_pago_com_subsidio:,.2f}")
print(f"\nValor total subsidiado pelo governo (Vl_Subsidio): R$ {valor_total_subsidiado:,.2f}")
print(f"\nValor total das tarifas cheias (Vl_Linha): R$ {valor_total_tarifa_cheia:,.2f}")
print(f"  (Diferença entre tarifa cheia e arrecadado, cobrida pelo subsídio + integrações)")

# 8. Passo: Percentuais
pct_subsidiado = (valor_total_subsidiado / valor_total_tarifa_cheia * 100) if valor_total_tarifa_cheia else 0
pct_integral = (valor_pago_integral / valor_total_arrecadado * 100) if valor_total_arrecadado else 0

print(f"\n=== PERCENTUAIS ===")
print(f"% da tarifa cheia cobertA pelo subsídio: {pct_subsidiado:.2f}%")
print(f"% do valor arrecadado vindo de pagamento integral: {pct_integral:.2f}%")

In [ ]:
# === ANÁLISE DESCRITIVA: Média diária de passageiros + Totais financeiros (médias diárias) ===

# 1. Passo: Total de viagens por dia
viagens_por_dia = Bu.groupby('data').size()

# 2. Passo: Calcular a média diária de viagens
media_diaria_viagens = viagens_por_dia.mean()

print("=== TOTAL DE VIAGENS POR DIA ===")
print(viagens_por_dia)

print(f"\n=== MÉDIA DIÁRIA DE PASSAGEIROS (VIAGENS) ===")
print(f"Média diária: {media_diaria_viagens:,.0f} viagens/dia")
print(f"Desvio-padrão entre os dias: {viagens_por_dia.std():,.0f}")
print(f"Dia com maior volume: {viagens_por_dia.idxmax()} ({viagens_por_dia.max():,} viagens)")
print(f"Dia com menor volume: {viagens_por_dia.idxmin()} ({viagens_por_dia.min():,} viagens)")

# 3. Passo: Totais financeiros por dia
financeiro_por_dia = Bu.groupby('data').agg(
    Total_Arrecadado=('Vl_Trans', 'sum'),
    Total_Subsidiado=('Vl_Subsidio', 'sum'),
    Total_Tarifa_Cheia=('Vl_Linha', 'sum')
)

# 4. Passo: Pago integralmente e pago com subsídio, por dia
pago_integral_por_dia = Bu[Bu['Grupo_Pagamento'] == 'Pagamento_Integral'].groupby('data')['Vl_Trans'].sum()
pago_com_subsidio_por_dia = Bu[Bu['Grupo_Pagamento'] == 'Com_Subsidio'].groupby('data')['Vl_Trans'].sum()

financeiro_por_dia['Pago_Integral'] = pago_integral_por_dia
financeiro_por_dia['Pago_Com_Subsidio'] = pago_com_subsidio_por_dia
financeiro_por_dia = financeiro_por_dia.fillna(0)

print("\n=== RESUMO FINANCEIRO POR DIA ===")
print(financeiro_por_dia.round(2))

# 5. Passo: Calcular as médias diárias de cada valor financeiro
media_arrecadado = financeiro_por_dia['Total_Arrecadado'].mean()
media_subsidiado = financeiro_por_dia['Total_Subsidiado'].mean()
media_tarifa_cheia = financeiro_por_dia['Total_Tarifa_Cheia'].mean()
media_pago_integral = financeiro_por_dia['Pago_Integral'].mean()
media_pago_com_subsidio = financeiro_por_dia['Pago_Com_Subsidio'].mean()

print(f"\n=== MÉDIA DIÁRIA - RESUMO FINANCEIRO ===")
print(f"Valor médio arrecadado por dia (Vl_Trans): R$ {media_arrecadado:,.2f}")
print(f"  - Pago integralmente (média/dia): R$ {media_pago_integral:,.2f}")
print(f"  - Pago pelo grupo com subsídio (média/dia): R$ {media_pago_com_subsidio:,.2f}")
print(f"\nValor médio subsidiado pelo governo por dia (Vl_Subsidio): R$ {media_subsidiado:,.2f}")
print(f"\nValor médio das tarifas cheias por dia (Vl_Linha): R$ {media_tarifa_cheia:,.2f}")

# 6. Passo: Percentuais (com base nas médias diárias)
pct_subsidiado = (media_subsidiado / media_tarifa_cheia * 100) if media_tarifa_cheia else 0
pct_integral = (media_pago_integral / media_arrecadado * 100) if media_arrecadado else 0

print(f"\n=== PERCENTUAIS (BASE: MÉDIA DIÁRIA) ===")
print(f"% da tarifa cheia média coberta pelo subsídio: {pct_subsidiado:.2f}%")
print(f"% do valor arrecadado médio vindo de pagamento integral: {pct_integral:.2f}%")

In [ ]:
# === VALOR MÉDIO POR VIAGEM (TICKET MÉDIO) - Pagamento Integral x Com Subsídio ===

ticket_medio_integral = Bu.loc[Bu['Grupo_Pagamento'] == 'Pagamento_Integral', 'Vl_Trans'].mean()
ticket_medio_subsidio = Bu.loc[Bu['Grupo_Pagamento'] == 'Com_Subsidio', 'Vl_Trans'].mean()

print("=== VALOR MÉDIO PAGO POR VIAGEM (TICKET MÉDIO) ===")
print(f"Pagamento Integral: R$ {ticket_medio_integral:,.2f} por viagem")
print(f"Com Subsídio:       R$ {ticket_medio_subsidio:,.2f} por viagem")

In [ ]:
# === ESTATÍSTICAS DESCRITIVAS - GRUPO PAGAMENTO INTEGRAL ===

dados_integral = Bu[Bu['Grupo_Pagamento'] == 'Pagamento_Integral']

media_vl_trans_integral = dados_integral['Vl_Trans'].mean()
mediana_vl_trans_integral = dados_integral['Vl_Trans'].median()
media_subsidio_integral = dados_integral['Vl_Subsidio'].mean()
desvio_subsidio_integral = dados_integral['Vl_Subsidio'].std()

print("=== PAGAMENTO INTEGRAL ===")
print(f"Tarifa média (Vl_Trans): R$ {media_vl_trans_integral:,.2f}")
print(f"Tarifa mediana (Vl_Trans): R$ {mediana_vl_trans_integral:,.2f}")
print(f"Média do subsídio (Vl_Subsidio): R$ {media_subsidio_integral:,.2f}")
print(f"Desvio padrão do subsídio (Vl_Subsidio): R$ {desvio_subsidio_integral:,.2f}")
print(dados_integral[['Vl_Linha', 'Vl_Trans', 'Vl_Subsidio']].describe().round(2))

In [ ]:
# === ESTATÍSTICAS DESCRITIVAS - GRUPO COM SUBSÍDIO ===

dados_subsidio = Bu[Bu['Grupo_Pagamento'] == 'Com_Subsidio']

media_vl_linha_subsidio = dados_subsidio['Vl_Linha'].mean()
media_vl_trans_subsidio = dados_subsidio['Vl_Trans'].mean()
mediana_vl_trans_subsidio = dados_subsidio['Vl_Trans'].median()
media_subsidio = dados_subsidio['Vl_Subsidio'].mean()
desvio_subsidio = dados_subsidio['Vl_Subsidio'].std()

print("=== COM SUBSÍDIO ===")
print(f"Custo médio nominal da linha (Vl_Linha): R$ {media_vl_linha_subsidio:,.2f}")
print(f"Valor médio transacionado (Vl_Trans): R$ {media_vl_trans_subsidio:,.2f}")
print(f"Valor mediano transacionado (Vl_Trans): R$ {mediana_vl_trans_subsidio:,.2f}")
print(f"Aporte médio de subsídio (Vl_Subsidio): R$ {media_subsidio:,.2f}")
print(f"Desvio padrão do subsídio (Vl_Subsidio): R$ {desvio_subsidio:,.2f}")
print(dados_subsidio[['Vl_Linha', 'Vl_Trans', 'Vl_Subsidio']].describe().round(2))

In [ ]:
# 1. Passo: Faixa de concentração (P25 - P75) geral
p25 = Bu['Vl_Trans'].quantile(0.25)
p75 = Bu['Vl_Trans'].quantile(0.75)
faixa_concentracao = Bu[(Bu['Vl_Trans'] >= p25) & (Bu['Vl_Trans'] <= p75)]
pct_concentracao = len(faixa_concentracao) / len(Bu) * 100

print("=== FAIXA DE CONCENTRAÇÃO GERAL (P25 - P75) ===")
print(f"Faixa: R$ {p25:.2f} a R$ {p75:.2f}")
print(f"Percentual de transações nessa faixa: {pct_concentracao:.2f}%")

# 2. Passo: Tarifas mais frequentes (moda) dentro da faixa, por Sindicato
print("\n=== TARIFA MAIS FREQUENTE (MODA) POR SINDICATO NA FAIXA DE CONCENTRAÇÃO ===")
moda_por_sindicato = faixa_concentracao.groupby('Sindicato')['Vl_Trans'].agg(
    Quantidade='count',
    Moda=lambda x: x.mode().iloc[0] if not x.mode().empty else None
).sort_values('Quantidade', ascending=False)
print(moda_por_sindicato.round(2))

# 3. Passo: Outliers extremos gerais (> R$ 50,00) por Sindicato
outliers_extremos = Bu[Bu['Vl_Trans'] > 50.00]

print("\n=== OUTLIERS EXTREMOS (Vl_Trans > R$ 50,00) POR SINDICATO ===")
print(outliers_extremos.groupby('Sindicato')['Vl_Trans'].agg(
    Quantidade='count',
    Media='mean',
    Maximo='max'
).round(2).sort_values('Maximo', ascending=False))

# 4. Passo: Pico máximo absoluto no grupo "Com_Subsidio" (elegível)
maximo_subsidio = Bu.loc[Bu['Grupo_Pagamento'] == 'Com_Subsidio', 'Vl_Trans'].max()
linha_pico = Bu.loc[(Bu['Grupo_Pagamento'] == 'Com_Subsidio') & (Bu['Vl_Trans'] == maximo_subsidio)]

print("\n=== PICO MÁXIMO NO GRUPO COM SUBSÍDIO ===")
print(f"Valor máximo: R$ {maximo_subsidio:,.2f}")
print(linha_pico[['Sindicato', 'Linha', 'Vl_Linha', 'Vl_Trans', 'Vl_Subsidio']].head())


# === ANÁLISE SEPARADA POR GRUPO: PAGAMENTO INTEGRAL x COM SUBSÍDIO ===

for grupo in ['Pagamento_Integral', 'Com_Subsidio']:
    dados_grupo = Bu[Bu['Grupo_Pagamento'] == grupo]

    print(f"\n\n{'='*60}")
    print(f"=== GRUPO: {grupo.upper()} ===")
    print(f"{'='*60}")

    # Faixa de concentração do grupo
    p25_g = dados_grupo['Vl_Trans'].quantile(0.25)
    p75_g = dados_grupo['Vl_Trans'].quantile(0.75)
    faixa_g = dados_grupo[(dados_grupo['Vl_Trans'] >= p25_g) & (dados_grupo['Vl_Trans'] <= p75_g)]
    pct_g = len(faixa_g) / len(dados_grupo) * 100

    print(f"Faixa de concentração (P25-P75): R$ {p25_g:.2f} a R$ {p75_g:.2f} "
          f"({pct_g:.2f}% das transações do grupo)")

    # Tarifa mais frequente do grupo (moda)
    moda_g = dados_grupo['Vl_Trans'].mode().iloc[0]
    print(f"Tarifa mais frequente (moda): R$ {moda_g:.2f}")

    # Modais predominantes na faixa de concentração, dentro do grupo
    print("\nModais predominantes na faixa de concentração:")
    print(faixa_g.groupby('Sindicato')['Vl_Trans'].agg(
        Quantidade='count',
        Moda=lambda x: x.mode().iloc[0] if not x.mode().empty else None
    ).sort_values('Quantidade', ascending=False).head(5).round(2))

    # Outliers extremos do grupo (> R$ 50,00)
    outliers_g = dados_grupo[dados_grupo['Vl_Trans'] > 50.00]
    print(f"\nOutliers extremos (> R$ 50,00): {len(outliers_g):,} registros")
    if len(outliers_g) > 0:
        print(outliers_g.groupby('Sindicato')['Vl_Trans'].agg(
            Quantidade='count', Media='mean', Maximo='max'
        ).round(2).sort_values('Maximo', ascending=False))

    # Máximo absoluto do grupo
    maximo_g = dados_grupo['Vl_Trans'].max()
    linha_max_g = dados_grupo[dados_grupo['Vl_Trans'] == maximo_g]
    print(f"\nValor máximo no grupo: R$ {maximo_g:,.2f}")
    print(linha_max_g[['Sindicato', 'Linha', 'Vl_Linha', 'Vl_Trans', 'Vl_Subsidio']].head(2))

In [ ]:
# === PAINEL DE GRÁFICOS: DISTRIBUIÇÃO DE FREQUÊNCIA E OUTLIERS ===

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Distribuição de Frequência e Outliers - Vl_Trans', fontsize=16, fontweight='bold')

# 1. Histograma geral com faixa de concentração (P25-P75) destacada
sns.histplot(Bu['Vl_Trans'], bins=40, kde=True, ax=axes[0, 0], color='#1f77b4')
axes[0, 0].axvspan(p25, p75, color='orange', alpha=0.25, label=f'Faixa P25-P75 (R$ {p25:.2f} - R$ {p75:.2f})')
axes[0, 0].axvline(p25, color='orange', linestyle='--', linewidth=1.5)
axes[0, 0].axvline(p75, color='orange', linestyle='--', linewidth=1.5)
axes[0, 0].set_title(f'Distribuição Geral de Vl_Trans\n{pct_concentracao:.1f}% das transações na faixa destacada')
axes[0, 0].set_xlabel('Vl_Trans (R$)')
axes[0, 0].set_ylabel('Frequência')
axes[0, 0].legend()

# 2. Boxplot por Sindicato com linha de referência em R$ 50,00
sns.boxplot(
    data=Bu, x='Sindicato', y='Vl_Trans',
    hue='Sindicato', palette='Set2', legend=False,
    fliersize=4,
    flierprops={"markerfacecolor": "red", "markeredgecolor": "red", "alpha": 0.5},
    ax=axes[0, 1]
)
axes[0, 1].axhline(50, color='red', linestyle='--', linewidth=1.5, label='Limiar de Outlier Extremo (R$ 50,00)')
axes[0, 1].set_title('Boxplot por Sindicato - Outliers Extremos')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('Vl_Trans (R$)')
axes[0, 1].tick_params(axis='x', rotation=20)
axes[0, 1].legend()
axes[0, 1].grid(axis='y', linestyle='--', alpha=0.5)

# 3 e 4. Histogramas comparando Pagamento Integral x Com Subsídio
for i, grupo in enumerate(['Pagamento_Integral', 'Com_Subsidio']):
    dados_grupo = Bu[Bu['Grupo_Pagamento'] == grupo]
    p25_g = dados_grupo['Vl_Trans'].quantile(0.25)
    p75_g = dados_grupo['Vl_Trans'].quantile(0.75)
    pct_g = len(dados_grupo[(dados_grupo['Vl_Trans'] >= p25_g) & (dados_grupo['Vl_Trans'] <= p75_g)]) / len(dados_grupo) * 100

    cor = '#2ca02c' if grupo == 'Pagamento_Integral' else '#d62728'

    sns.histplot(dados_grupo['Vl_Trans'], bins=40, kde=True, ax=axes[1, i], color=cor)
    axes[1, i].axvspan(p25_g, p75_g, color='orange', alpha=0.25, label=f'Faixa P25-P75 (R$ {p25_g:.2f} - R$ {p75_g:.2f})')
    axes[1, i].axvline(p25_g, color='orange', linestyle='--', linewidth=1.5)
    axes[1, i].axvline(p75_g, color='orange', linestyle='--', linewidth=1.5)
    axes[1, i].set_title(f'{grupo}\n{pct_g:.1f}% das transações na faixa destacada')
    axes[1, i].set_xlabel('Vl_Trans (R$)')
    axes[1, i].set_ylabel('Frequência')
    axes[1, i].legend()

plt.tight_layout()
plt.savefig('imgs/distribuicao_frequencia_outliers_painel.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

In [ ]:
# 1. Passo: Definir o tamanho do painel do gráfico
plt.figure(figsize=(11, 6))

# 2. Passo: Desenhar o Boxplot por Sindicato (mantido)
sns.boxplot(
    data=Bu, 
    x='Sindicato', 
    y='Vl_Linha', 
    hue='Sindicato',
    palette='Set2',
    legend=False,
    fliersize=5,
    flierprops={"markerfacecolor": "red", "markeredgecolor": "red", "alpha": 0.5}
)

plt.title('Identificação de Outliers - Estudo Estatístico de Tarifas por Modal (Sindicato)', fontsize=14, fontweight='bold')
plt.xlabel('Modal de Transporte (Agrupamento por Sindicato/Federação)')
plt.ylabel('Tarifa Nominal da Linha (R$)')
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('imgs/outliers_sindicato.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

# === NOVO: Checagem de outliers por dia (detectar dia com importação anômala) ===
plt.figure(figsize=(11, 6))

sns.boxplot(
    data=Bu,
    x='data',
    y='Vl_Trans',
    hue='data',
    palette='Set3',
    legend=False,
    fliersize=4,
    flierprops={"markerfacecolor": "red", "markeredgecolor": "red", "alpha": 0.5}
)

plt.title('Identificação de Outliers - Valor Pago (Vl_Trans) por Dia', fontsize=14, fontweight='bold')
plt.xlabel('Data')
plt.ylabel('Valor Pago pelo Usuário (R$)')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('imgs/outliers_por_dia.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

In [ ]:
# === LISTAGEM DOS VALORES ÚNICOS DE SINDICATO ===
print("=== VALORES ÚNICOS DE SINDICATO (com contagem) ===")
print(Bu['Sindicato'].value_counts())

In [ ]:
# === ANÁLISE DE IMPACTO SOCIAL: TRANSPORTE DE MASSA x SEGMENTADO ===

# 1. Passo: Classificação dos modais
modais_massa = ['SUPERVIA', 'METRÔ RIO', 'BARCAS']
modais_segmentado = ['TRANSONIBUS', 'SETRERJ', 'SETRANSDUC', 'VANS INTERMUNICIPAIS',
                      'SINTERJ', 'RIO ÔNIBUS', 'SETRANSOL', 'NÃO FILIADOS']

def classificar_modal(sindicato):
    if sindicato in modais_massa:
        return 'Transporte de Massa'
    elif sindicato in modais_segmentado:
        return 'Transporte Segmentado'
    else:
        return 'Não Classificado'

Bu['Tipo_Modal'] = Bu['Sindicato'].apply(classificar_modal)

print("=== DISTRIBUIÇÃO POR TIPO DE MODAL ===")
print(Bu['Tipo_Modal'].value_counts())


# === INDICADOR 1: TARIFA E SUBSÍDIO (Vl_Trans, Vl_Subsidio, Vl_Linha) ===
print("\n\n=== INDICADOR 1: ESTATÍSTICAS TARIFÁRIAS E DE SUBSÍDIO ===")
resumo_tarifario = Bu.groupby('Tipo_Modal')[['Vl_Linha', 'Vl_Trans', 'Vl_Subsidio']].agg(['mean', 'median', 'std', 'max']).round(2)
print(resumo_tarifario)

# Boxplots comparativos
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Impacto Social - Transporte de Massa x Segmentado: Indicadores Tarifários', fontsize=15, fontweight='bold')

for i, coluna in enumerate(['Vl_Linha', 'Vl_Trans', 'Vl_Subsidio']):
    sns.boxplot(
        data=Bu[Bu['Tipo_Modal'] != 'Não Classificado'],
        x='Tipo_Modal',
        y=coluna,
        hue='Tipo_Modal',
        palette={'Transporte de Massa': '#1f77b4', 'Transporte Segmentado': '#ff7f0e'},
        legend=False,
        fliersize=4,
        flierprops={"markerfacecolor": "red", "markeredgecolor": "red", "alpha": 0.5},
        ax=axes[i]
    )
    axes[i].set_title(coluna)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Valor (R$)')
    axes[i].grid(axis='y', linestyle='--', alpha=0.5)
    axes[i].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig('imgs/impacto_social_tarifario.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()


# === INDICADOR 2: VOLUME DE VIAGENS (escala do impacto) ===
print("\n\n=== INDICADOR 2: VOLUME DE VIAGENS POR TIPO DE MODAL ===")
volume_modal = Bu.groupby('Tipo_Modal').agg(
    Total_Viagens=('Data_da_Transacao', 'count'),
    Cartoes_Unicos=('Cartao_Hash', 'nunique')
)
volume_modal['Pct_Viagens'] = (volume_modal['Total_Viagens'] / volume_modal['Total_Viagens'].sum() * 100).round(2)
print(volume_modal)

plt.figure(figsize=(9, 5))
volume_plot = volume_modal[volume_modal.index != 'Não Classificado']
plt.bar(volume_plot.index, volume_plot['Total_Viagens'],
        color=['#1f77b4', '#ff7f0e'], alpha=0.85)
plt.title('Volume de Viagens - Transporte de Massa x Segmentado', fontsize=14, fontweight='bold')
plt.ylabel('Quantidade de Viagens')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('imgs/impacto_social_volume.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()


# === INDICADOR 3: EFICIÊNCIA DO SUBSÍDIO POR PASSAGEIRO/VIAGEM ===
print("\n\n=== INDICADOR 3: EFICIÊNCIA DO SUBSÍDIO POR MODAL ===")

eficiencia_subsidio = Bu.groupby('Tipo_Modal').agg(
    Total_Viagens=('Data_da_Transacao', 'count'),
    Subsidio_Total=('Vl_Subsidio', 'sum'),
    Subsidio_Medio_Por_Viagem=('Vl_Subsidio', 'mean'),
    Pct_Viagens_Subsidiadas=('Vl_Subsidio', lambda x: (x > 0).mean() * 100)
).round(2)

print(eficiencia_subsidio)

# Custo do subsídio por viagem - gráfico comparativo
plt.figure(figsize=(9, 5))
eficiencia_plot = eficiencia_subsidio[eficiencia_subsidio.index != 'Não Classificado']
plt.bar(eficiencia_plot.index, eficiencia_plot['Subsidio_Medio_Por_Viagem'],
        color=['#1f77b4', '#ff7f0e'], alpha=0.85)
plt.title('Subsídio Médio por Viagem - Transporte de Massa x Segmentado', fontsize=14, fontweight='bold')
plt.ylabel('Subsídio Médio (R$)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('imgs/impacto_social_eficiencia_subsidio.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()


# === SÍNTESE: CUSTO TOTAL DE SUBSÍDIO POR MILHÃO DE VIAGENS ===
print("\n\n=== SÍNTESE: CUSTO DE SUBSÍDIO POR CADA 1 MILHÃO DE VIAGENS ===")
sintese = eficiencia_subsidio[eficiencia_subsidio.index != 'Não Classificado'].copy()
sintese['Subsidio_Por_Milhao_Viagens'] = (sintese['Subsidio_Total'] / sintese['Total_Viagens'] * 1_000_000).round(2)
print(sintese[['Total_Viagens', 'Subsidio_Total', 'Subsidio_Por_Milhao_Viagens']])

In [ ]:
# === OPERADORA COM MAIS TRANSAÇÕES ===

ranking_operadora = Bu['Operadora'].value_counts()

print("=== RANKING DE TRANSAÇÕES POR OPERADORA (TOP 10) ===")
print(ranking_operadora.head(10))

print(f"\nOperadora com mais transações: {ranking_operadora.idxmax()} "
      f"({ranking_operadora.max():,} transações)")

In [ ]:
# === OPERADORA COM MAIOR TARIFA NOMINAL (Vl_Linha) ===

tarifa_por_operadora = Bu.groupby('Operadora')['Vl_Linha'].agg(
    Media='mean',
    Mediana='median',
    Maximo='max',
    Quantidade='count'
).round(2).sort_values('Media', ascending=False)

print("=== TARIFA NOMINAL (Vl_Linha) POR OPERADORA - TOP 10 (MAIOR MÉDIA) ===")
print(tarifa_por_operadora.head(10))

print(f"\nOperadora com maior tarifa nominal média: {tarifa_por_operadora.index[0]} "
      f"(R$ {tarifa_por_operadora['Media'].iloc[0]:,.2f})")

print(f"\nOperadora com maior tarifa nominal máxima registrada: "
      f"{tarifa_por_operadora['Maximo'].idxmax()} "
      f"(R$ {tarifa_por_operadora['Maximo'].max():,.2f})")

print(f"\nOperadora com maior tarifa nominal média: {tarifa_por_operadora.index[0]} "
f"(R$ {tarifa_por_operadora['Media'].iloc[0]:,.2f})")

In [ ]:
# === OPERADORA QUE SE DESTACA EM TARIFA NOMINAL MÉDIA (Vl_Linha) ===

tarifa_por_operadora = Bu.groupby('Operadora')['Vl_Linha'].agg(
    Media='mean',
    Mediana='median',
    Maximo='max',
    Quantidade='count'
).round(2).sort_values('Media', ascending=False)

print("=== RANKING COMPLETO - TARIFA NOMINAL MÉDIA POR OPERADORA ===")
print(tarifa_por_operadora)

# Destaque: operadora com maior média
top_operadora = tarifa_por_operadora.index[0]
top_media = tarifa_por_operadora['Media'].iloc[0]
media_geral = Bu['Vl_Linha'].mean()

print(f"\n=== OPERADORA QUE MAIS SE DESTACA ===")
print(f"Operadora: {top_operadora}")
print(f"Tarifa nominal média: R$ {top_media:,.2f}")
print(f"Média geral do sistema (todas operadoras): R$ {media_geral:,.2f}")
print(f"Diferença em relação à média geral: {top_media - media_geral:+,.2f} "
      f"({(top_media / media_geral - 1) * 100:+.1f}%)")
      

In [ ]:
# === OUTLIERS DE TARIFA (Vl_Trans) POR TURNO - LIMIAR BASEADO EM IQR ===

# 1. Passo: Calcular o limiar de outlier via IQR (Q1, Q3, IQR) sobre a base geral
Q1 = Bu['Vl_Trans'].quantile(0.25)
Q3 = Bu['Vl_Trans'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

print("=== LIMIAR DE OUTLIER (BASEADO EM IQR) ===")
print(f"Q1 = R$ {Q1:.2f} | Q3 = R$ {Q3:.2f} | IQR = R$ {IQR:.2f}")
print(f"Limite superior (Q3 + 1.5*IQR) = R$ {limite_superior:.2f}")

# 2. Passo: Filtrar os outliers segundo esse limiar
outliers_iqr = Bu[Bu['Vl_Trans'] > limite_superior]

print(f"\nTotal de outliers (Vl_Trans > R$ {limite_superior:.2f}): {len(outliers_iqr):,} "
      f"({len(outliers_iqr)/len(Bu)*100:.2f}% da base)")

# 3. Passo: Quantidade, média e máximo dos outliers por turno
print("\n=== OUTLIERS POR TURNO ===")
outliers_por_turno = outliers_iqr.groupby('Turno').agg(
    Quantidade=('Vl_Trans', 'count'),
    Media=('Vl_Trans', 'mean'),
    Maximo=('Vl_Trans', 'max')
).round(2)
print(outliers_por_turno)

# Percentual que cada turno representa do total de outliers
total_outliers = outliers_por_turno['Quantidade'].sum()
outliers_por_turno['Pct_dos_Outliers'] = (outliers_por_turno['Quantidade'] / total_outliers * 100).round(2)

print(f"\nTotal de outliers: {total_outliers:,}")
print(outliers_por_turno[['Quantidade', 'Pct_dos_Outliers']])

# 4. Passo: Taxa de incidência - outliers como % do volume total de cada turno
volume_total_por_turno = Bu.groupby('Turno').size()
taxa_incidencia = (outliers_por_turno['Quantidade'] / volume_total_por_turno * 100).round(4)

print("\n=== TAXA DE INCIDÊNCIA DE OUTLIERS POR TURNO (% do volume total do turno) ===")
print(taxa_incidencia)

turno_maior_incidencia = taxa_incidencia.idxmax()
print(f"\nTurno com maior proporção de outliers: {turno_maior_incidencia} "
      f"({taxa_incidencia.max():.4f}% das viagens desse turno)")

# 5. Passo: Gráfico - boxplot de Vl_Trans por turno, com o limiar IQR destacado
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=Bu, x='Turno', y='Vl_Trans',
    hue='Turno', palette='Set2', legend=False,
    fliersize=4,
    flierprops={"markerfacecolor": "red", "markeredgecolor": "red", "alpha": 0.5}
)
plt.axhline(limite_superior, color='red', linestyle='--', linewidth=1.5,
            label=f'Limiar de Outlier (Q3 + 1.5×IQR = R$ {limite_superior:.2f})')
plt.title('Distribuição de Vl_Trans por Turno - Identificação de Outliers (IQR)', fontsize=14, fontweight='bold')
plt.xlabel('Turno Operacional')
plt.ylabel('Vl_Trans (R$)')
plt.xticks(rotation=15)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('imgs/outliers_por_turno_iqr.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close()

In [ ]:
# 1. Filtrar a base apenas para o grupo que recebe subsídio
# (Ignoramos as gratuidades e o pagamento integral para não zerar a conta)
dados_subsidio = Bu[Bu["Perfil"] == "Subsidiado"].copy()

# 2. Definir as variáveis da Regressão
# Queremos prever o Subsídio (Y) baseado no Valor Total da Linha (X)
X = dados_subsidio[["Vl_Linha"]]  # Variável independente (X deve ser 2D)
y = dados_subsidio["Vl_Subsidio"]  # Variável dependente/alvo

# 3. Importar as funções necessárias do scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 4. Criar e treinar o modelo
modelo_subsidio = LinearRegression()
modelo_subsidio.fit(X, y)

# 5. Gerar as previsões para a base
y_pred = modelo_subsidio.predict(X)

# 6. Exibir os resultados na tela
print("=" * 70)
print("REGRESSÃO LINEAR: IMPACTO DO VALOR DA LINHA NO SUBSÍDIO")
print("=" * 70)
print(f"Coeficiente Angular (Inclinação): {modelo_subsidio.coef_[0]:.4f}")
print(f"Intercepto (Eixo Y): {modelo_subsidio.intercept_:.4f}")
print(f"R² (Poder de explicação do modelo): {r2_score(y, y_pred):.4f}")
print(
    f"Erro Quadrático Médio (MSE): {mean_squared_error(y, y_pred):.4f}\n"
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15, 6))

# 1. Pegamos uma amostra aleatória de ambos os grupos para o gráfico não travar o computador
# (Plotar 3 milhões de pontos de uma vez pesaria demais na memória)
amostra_integral = Bu[Bu["Perfil"] == "Pagamento_Integral"].sample(
    n=3000, random_state=42
)
amostra_subsidio = Bu[Bu["Perfil"] == "Subsidiado"].sample(
    n=3000, random_state=42
)

# 2. Plotar os pontos reais das duas categorias
plt.scatter(
    amostra_integral["Vl_Linha"],
    amostra_integral["Vl_Subsidio"],
    alpha=0.3,
    color="gray",
    label="Pagamento Integral (Subsídio = 0)",
)
plt.scatter(
    amostra_subsidio["Vl_Linha"],
    amostra_subsidio["Vl_Subsidio"],
    alpha=0.4,
    color="blue",
    label="Grupo Com Subsídio (Pontos Reais)",
)

# 3. Plotar a linha vermelha da sua Regressão Linear por cima
plt.plot(
    X,
    y_pred,
    color="red",
    linewidth=2.5,
    label="Linha de Tendência do Subsídio (Modelo)",
)

# Configurações visuais do gráfico
plt.title(
    "Regressão Linear: Relação entre Preço da Linha e Subsídio Pago",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Valor Total da Linha (R$)")
plt.ylabel("Valor do Subsídio Pago (R$)")
plt.legend(loc="upper left")
plt.grid(True, linestyle="--", alpha=0.5)

# Salva na sua pasta de imagens
plt.savefig("imgs/regressao_subsidio.png", bbox_inches="tight", dpi=150)
plt.show()

## **6. MODELAGEM ESTATÍSTICA: REGRESSÃO LINEAR MULTIGRUPO**

Para aprofundar o entendimento sobre a dinâmica econômico-financeira do Sistema de Bilhetagem Eletrônica (RioCard), aplicamos um modelo de **Regressão Linear Simples** por meio da biblioteca `scikit-learn`. O objetivo principal foi isolar o comportamento dos dois grupos comerciais majoritários identificados na base de dados no campo `Perfil`: o grupo de **Pagamento_Integral** e o grupo **Subsidiado** (que recebe aporte governamental).

### **6.1 Definição do Problema e Premissas**
Enquanto o grupo de *Pagamento_Integral* possui uma relação estática e nula em relação ao subsídio (onde $Vl\_Subsidio = 0$ para qualquer valor de tarifa), o grupo de usuários com perfil *Subsidiado* apresenta uma variação que impacta diretamente os cofres públicos. 

A modelagem foi configurada para responder à seguinte questão de pesquisa: **"Para cada R$ 1,00 de aumento no valor nominal de uma linha de transporte ($Vl\_Linha$), qual é o impacto médio gerado no valor do subsídio ($Vl\_Subsidio$) custeado pelo governo?"**

* **Variável Independente ($X$):** `Vl_Linha` (Valor total da tarifa da linha)
* **Variável Dependente ($Y$):** `Vl_Subsidio` (Valor do subsídio pago pelo poder público)

---

### **6.2 Resultados Obtidos pelo Modelo**

Após o treinamento do algoritmo com a totalidade dos registros do grupo mapeado, o modelo reportou os seguintes parâmetros estatísticos exatos:

* **Coeficiente Angular (Inclinação da Reta):** `0.6499`
* **Intercepto (Coeficiente Linear):** `-2.1285`
* **Coeficiente de Determinação ($R^2$):** `0.6601`
* **Erro Quadrático Médio (MSE):** `4.2928`

A partir desses coeficientes, obtivemos a seguinte equação preditiva para o perfil de usuários subsidiados:

$$\text{Valor do Subsídio (R\$)} = (0.6499 \times \text{Valor da Linha}) - 2.1285$$

---

### **6.3 Análise e Interpretação dos Resultados**

1. **Aderência do Modelo ($R^2$):** O coeficiente de determinação de `0.6601` demonstra que o modelo possui uma **alta** capacidade de explicação para um modelo de linha única. Isso significa que aproximadamente **66,01%** da variação do valor do subsídio praticado no sistema de transporte é determinada diretamente pelo valor cheio da tarifa fixada para a linha ($Vl\_Linha$). O restante da variação (33,99%) se deve a regras específicas de integração tarifária entre diferentes modais e baldeações temporais que fogem à linearidade simples.

2. **Impacto Econômico (Coeficiente Angular):** O coeficiente angular de `0.6499` revela que, para o grupo de usuários beneficiados, cada aumento de **R$ 1,00** no valor total das passagens ($Vl\_Linha$) resulta em um acréscimo automático e médio de **R$ 0,65** no montante do subsídio custeado pelo Estado. O intercepto negativo (`-2.1285`) indica matematicamente o ponto de corte inicial a partir do qual as políticas de subsídio passam a ser aplicadas de forma efetiva sobre o valor das tarifas.

3. **Conclusão de Big Data:** A dispersão observada no conjunto de dados e a reta de regressão calculada comprovam que o subsídio não é um valor fixo por usuário, mas sim uma variável fortemente indexada ao valor da tarifa da linha. Para a gestão de Big Data em políticas públicas, este modelo serve como uma ferramenta preditiva eficiente: ele permite que a Secretaria de Transportes calcule o impacto financeiro estimado de um futuro reajuste de tarifas sobre o orçamento municipal, garantindo a previsibilidade dos subsídios necessários para manter a modicidade tarifária dos passageiros com direito ao benefício.